In [9]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.schema import Document


In [10]:
docs = [

Document (page_content="LangChain helps build LLM applications."),

Document (page_content="Pinecone is a vector database for semantic search."),

Document (page_content="The Eiffel Tower is located in Paris."),

Document (page_content="Langchain can be used to develop agentic ai application."),

Document (page_content="Langchain has many types of retrievers. ")

]

embedding_model = HuggingFaceBgeEmbeddings(model_name = "all-MiniLM-L6-v2")
dense_vectorstore = FAISS.from_documents(docs, embedding_model)
dense_retriever = dense_vectorstore.as_retriever()




Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6696.84it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
sparse_retriever  = BM25Retriever.from_documents(docs)
sparse_retriever.k = 3

hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever,sparse_retriever],
    weight = [0.6,0.4]
)

In [12]:
hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceBgeEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x13160fb10>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x131702b50>, k=3)], weights=[0.5, 0.5])

In [13]:
query = "How to build LLM"

results = hybrid_retriever.invoke(query)

for i,doc in enumerate(results):
    print(f"document {i+1}: \n {doc.page_content}")

document 1: 
 LangChain helps build LLM applications.
document 2: 
 Langchain can be used to develop agentic ai application.
document 3: 
 Langchain has many types of retrievers. 
document 4: 
 Pinecone is a vector database for semantic search.


In [14]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from langchain_classic.schema import Document
from langchain_classic.vectorstores import FAISS
from langchain_classic.embeddings import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.schema.runnable import RunnableLambda, RunnableMap
from langchain_classic.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [25]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",temperature=0.2)

prompt = PromptTemplate.from_template(
    """"Answer the Question based on the context below 
    Context:
    {context}

    Question: {input}
    """
)

In [26]:
document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)

In [27]:
rag_chain = create_retrieval_chain(
    retriever=hybrid_retriever,
    combine_docs_chain = document_chain,
    )

In [28]:
query = {"input":"How can i build LLM application"}

response = rag_chain.invoke(query)

print(f"Answer:\n", response["answer"])

Answer:
 LangChain helps build LLM applications.
